# Week 2: CNN/RNN/Optimizer

Notes from MIT 6.S191 lectures 1 & 2.

In [ ]:
import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.CIFAR10('data', train=True, download=True, transform=transform)
test_set = datasets.CIFAR10('data', train=False, download=True, transform=transform)

def get_loaders(batch_size):
    return (torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True),
            torch.utils.data.DataLoader(test_set, batch_size=1000))

In [ ]:
class E2ENet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 24, 5, 2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, 2), nn.ReLU(),
            nn.Conv2d(36, 48, 5, 2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
            nn.Conv2d(64, 64, 3), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*1*1, 100), nn.ReLU(),
            nn.Linear(100, 50), nn.ReLU(),
            nn.Linear(50, 10)
        )
    def forward(self, x):
        return self.fc(self.conv(x))

In [ ]:
def train(lr, batch_size, epochs=1):
    model = E2ENet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    train_loader, test_loader = get_loaders(batch_size)
    writer = SummaryWriter(f'runs/lr{lr}_bs{batch_size}')
    step = 0
    for epoch in range(epochs):
        model.train()
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            opt.zero_grad()
            out = model(data)
            loss = crit(out, target)
            loss.backward()
            opt.step()
            writer.add_scalar('loss', loss.item(), step)
            step += 1
    writer.close()
    return model

In [ ]:
for lr in [1e-3, 1e-4]:
    for bs in [64, 128]:
        train(lr, bs, epochs=1)